In [1]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3
import os

I0000 00:00:1785263164.110077   28589 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Segmentation Models: using `tf.keras` framework.


In [ ]:
# run one time ...
src = "data/CTA nii"
pname = os.listdir(src)
cnt = 0
for name in pname :
    res = tts.merge(os.path.join(src , name))
    cnt+=1
else :
    print("done !!!")

In [2]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet , mergeTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet  , mergeTestSet  = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet , mergeTrainSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet , mergeTestSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")

In [3]:
# pipeline configuring

geo      = utl.randomGeo(p=0.7)
crop     = utl.volume_crop((128 , 128 , 128) , paddDim=80)
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 128 , 128 , 128 , 3] ,
    labelShape=[None , 128 , 128 , 128 , 1]
)

windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=0.7 ,
    p_ww=0.7
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64 , tf.float64])
def rimg(imgPath , labelPath , mergePath) :
    return utl.read_img(imgPath , labelPath , mergePath)
def read_img(img , label , merge) :
    imglbl = rimg(img , label , merge)
    img   = imglbl[0]
    label = imglbl[1]
    merge = imglbl[2]
    return img , label , merge

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



I0000 00:00:1785263175.280488   28589 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1785263175.882359   28589 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1785263175.887439   28589 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1

In [ ]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet , mergeTrainSet))
    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )

    .cache("myCacheTrain")
    .shuffle(buffer_size=20 , seed=42 , reshuffle_each_iteration=True)
    
    .map(
        rot ,
        num_parallel_calls=2
    )
    .map(
        geo.flipX ,
        num_parallel_calls=2
    )
    .map(
        geo.flipY ,
        num_parallel_calls=2
    )
    .map(
        geo.flipZ ,
        num_parallel_calls=2
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=2
    )

    .batch(batch_size=4)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTestSet , labelTestSet , mergeTestSet))

    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )
    .map(
        utl.channelize ,
        num_parallel_calls=4
    )
    .cache("myCacheValid")
    .batch(batch_size=2)
    .map(
        windower.apply_default , 
        num_parallel_calls=4
    )
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
for data in dataloaderValid.take(5) :
    print("image shape :" , data[0].shape)
    print("label shape :" , data[1].shape)

In [5]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4_res18
)


# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze_Conv3_unit1ToEnd_res18 , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)


In [ ]:
print(model.summary())

In [11]:
# compilation
lr = keras.optimizers.schedules.PiecewiseConstantDecay(
    [
        5530 ,
        11060 ,

    ] ,
    [
        1e-3 ,
        1e-4 ,
        1e-5
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice
    ] ,
)

In [ ]:
logger = keras.callbacks.CSVLogger("tuning_logs/technical.csv")

# model training
history = model.fit(
    x = dataloaderTrain ,
    epochs=250 ,
    validation_data = dataloaderValid ,
    callbacks=[
        logger
    ]

)